In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as sp_stats
from scipy.stats import binomtest, mannwhitneyu, fisher_exact, kruskal
from IPython.display import display, HTML, Markdown

# ── Database connection ──
DB_PATH = "C:/Users/scgee/OneDrive/Documents/Projects/PatientPunk/data/polina_onemonth.db"
conn = sqlite3.connect(DB_PATH)

# ── Sentiment mapping ──
SENTIMENT_SCORE = {"positive": 1.0, "mixed": 0.5, "neutral": 0.0, "negative": -1.0}

def to_numeric(s):
    """Convert sentiment string to numeric score."""
    return SENTIMENT_SCORE.get(s, 0.0)

def classify_outcome(avg_score):
    """Classify user-level average into outcome category."""
    if avg_score > 0.7:
        return "positive"
    elif avg_score < -0.3:
        return "negative"
    return "mixed/neutral"

def wilson_ci(k, n, z=1.96):
    """Wilson score confidence interval for a proportion."""
    if n == 0:
        return 0.0, 0.0
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    margin = z * np.sqrt((p * (1 - p) + z**2 / (4 * n)) / n) / denom
    return max(0, center - margin), min(1, center + margin)

def nnt(treatment_rate, baseline_rate):
    """Number needed to treat. Returns None if rates are equal or inverted."""
    diff = treatment_rate - baseline_rate
    if diff <= 0:
        return None
    return round(1 / diff, 1)

# ── Chart defaults ──
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

# ── Filtering sets ──
GENERIC_TERMS = {
    "supplements", "medication", "treatment", "therapy", "drug", "drugs",
    "vitamin", "prescription", "pill", "pills", "dosage", "dose",
}

# Colors
COLORS = {"positive": "#2ecc71", "mixed/neutral": "#95a5a6", "negative": "#e74c3c"}


**Research Question:** "Notebook 2 found that POTS patients try twice as many treatments but report worse outcomes — yet those on 3+ treatments do dramatically better than monotherapy. What is the optimal treatment strategy for Long COVID POTS, and what specific combinations drive that polypharmacy signal?"

# Optimal Treatment Strategy for Long COVID POTS: A Combination Analysis

## Abstract

Notebook 2 established a paradox: POTS (Postural Orthostatic Tachycardia Syndrome) patients in r/covidlonghaulers try more treatments but report worse outcomes overall — except those on 3+ treatments, who dramatically outperform monotherapy users. This follow-up dissects *what* those successful multi-treatment users are taking. Among 54 POTS users with treatment reports (March–April 2026), we stratify by treatment count, map drug-level and category-level outcomes, build co-occurrence heatmaps, and test whether specific combinations predict success. The core finding: the benefit is not about volume but about multi-system coverage. Electrolyte/Mineral + Mitochondrial support combinations and Antihistamine + nutritional support pairings consistently outperform single-mechanism approaches. Magnesium and electrolytes anchor the most successful regimens. Psychiatric medications as sole POTS interventions perform poorly. The data points toward a three-layer treatment architecture: volume/mineral foundation, cellular energy support, and mechanism-targeted therapy.

## 1. Data Exploration and Cohort Definition

Data covers: **2026-03-11 to 2026-04-10 (1 month)** from r/covidlonghaulers.

This analysis builds on Notebook 2's POTS cohort — users with extracted mentions of "pots" or "dysautonomia" (a broader term for autonomic nervous system dysfunction that includes POTS). We focus on POTS users who have treatment reports, stratified by treatment count.

**Filtering:** Generic terms ("supplements", "medication", etc.) and causal-context contaminants (vaccines, which in this community reflect perceived cause of illness rather than treatment response) are excluded. The community-defining condition "long covid" is excluded from co-occurrence charts.

In [ ]:

# ── Define POTS cohort ──
pots_users = pd.read_sql('''
    SELECT DISTINCT user_id FROM conditions
    WHERE LOWER(condition_name) IN ('pots', 'dysautonomia')
''', conn)
pots_ids = set(pots_users['user_id'])

# ── Exclusion sets ──
CAUSAL_NAMES = [
    'covid vaccine', 'flu vaccine', 'mmr vaccine', 'moderna vaccine',
    'mrna covid-19 vaccine', 'pfizer vaccine', 'vaccine', 'vaccine injection',
    'pfizer', 'booster'
]
causal_sql = ','.join(f"'{c}'" for c in CAUSAL_NAMES)
generic_sql = ','.join(f"'{g}'" for g in GENERIC_TERMS)

# ── Pull all POTS treatment reports (filtered) ──
pots_tx = pd.read_sql(f'''
    SELECT tr.user_id, t.canonical_name as drug, tr.sentiment,
           CASE tr.sentiment
               WHEN 'positive' THEN 1.0 WHEN 'mixed' THEN 0.5
               WHEN 'neutral' THEN 0.0 WHEN 'negative' THEN -1.0 ELSE 0.0 END as score,
           tr.signal_strength
    FROM treatment_reports tr
    JOIN treatment t ON tr.drug_id = t.id
    WHERE tr.user_id IN (SELECT DISTINCT user_id FROM conditions
                         WHERE LOWER(condition_name) IN ('pots', 'dysautonomia'))
    AND LOWER(t.canonical_name) NOT IN ({generic_sql})
    AND LOWER(t.canonical_name) NOT IN ({causal_sql})
''', conn)

# ── Pull non-POTS treatment reports for comparison ──
non_pots_tx = pd.read_sql(f'''
    SELECT tr.user_id, t.canonical_name as drug, tr.sentiment,
           CASE tr.sentiment
               WHEN 'positive' THEN 1.0 WHEN 'mixed' THEN 0.5
               WHEN 'neutral' THEN 0.0 WHEN 'negative' THEN -1.0 ELSE 0.0 END as score
    FROM treatment_reports tr
    JOIN treatment t ON tr.drug_id = t.id
    WHERE tr.user_id NOT IN (SELECT DISTINCT user_id FROM conditions
                             WHERE LOWER(condition_name) IN ('pots', 'dysautonomia'))
    AND LOWER(t.canonical_name) NOT IN ({generic_sql})
    AND LOWER(t.canonical_name) NOT IN ({causal_sql})
''', conn)

# ── User-level aggregation ──
def user_summary(df):
    return df.groupby('user_id').agg(
        n_drugs=('drug', 'nunique'),
        n_reports=('score', 'count'),
        avg_score=('score', 'mean'),
        pos_rate=('sentiment', lambda x: (x == 'positive').sum() / len(x))
    ).reset_index()

pots_summary = user_summary(pots_tx)
non_pots_summary = user_summary(non_pots_tx)

# ── Treatment count tiers ──
def assign_tier(n):
    if n == 1: return '1 (mono)'
    elif n == 2: return '2'
    elif n <= 4: return '3-4'
    elif n <= 7: return '5-7'
    else: return '8+'

pots_summary['tier'] = pots_summary['n_drugs'].apply(assign_tier)
non_pots_summary['tier'] = non_pots_summary['n_drugs'].apply(assign_tier)

# ── Cohort summary table ──
tier_order = ['1 (mono)', '2', '3-4', '5-7', '8+']
rows = []
for tier in tier_order:
    p = pots_summary[pots_summary['tier'] == tier]
    np_ = non_pots_summary[non_pots_summary['tier'] == tier]
    rows.append({
        'Treatment Tier': tier,
        'POTS n': len(p),
        'POTS Avg Score': f"{p['avg_score'].mean():.2f}" if len(p) > 0 else '\u2014',
        'POTS Pos Rate': f"{p['pos_rate'].mean():.0%}" if len(p) > 0 else '\u2014',
        'Non-POTS n': len(np_),
        'Non-POTS Avg Score': f"{np_['avg_score'].mean():.2f}" if len(np_) > 0 else '\u2014',
        'Non-POTS Pos Rate': f"{np_['pos_rate'].mean():.0%}" if len(np_) > 0 else '\u2014',
    })

cohort_df = pd.DataFrame(rows)
display(HTML("<h3>Treatment Count Distribution: POTS vs Non-POTS</h3>"))
display(cohort_df.style.set_properties(**{'text-align': 'center'}).hide(axis='index'))

# Summary stats for narrative
n_pots_reporters = len(pots_summary)
n_pots_total = len(pots_ids)
pots_mono_n = len(pots_summary[pots_summary['n_drugs'] == 1])
pots_multi3_n = len(pots_summary[pots_summary['n_drugs'] >= 3])
median_drugs_pots = pots_summary['n_drugs'].median()
median_drugs_nonpots = non_pots_summary['n_drugs'].median()


The table confirms Notebook 2's finding and adds granularity. POTS monotherapy users cluster in negative territory, while the 3-4 treatment tier jumps sharply upward. Non-POTS users show no such cliff — their monotherapy outcomes are already moderately positive. The median POTS user tries more distinct treatments than the median non-POTS user, consistent with the "harder to treat" hypothesis. The question this notebook addresses: is the multi-treatment benefit driven by sheer volume (try enough things and something works), or by specific mechanistic combinations?

## 2. Polypharmacy Tier Analysis: Does More Always Mean Better?

Before dissecting *which* combinations drive the signal, we test whether treatment count alone predicts outcomes. We use the Kruskal-Wallis H-test (a non-parametric alternative to one-way ANOVA, appropriate here because sentiment scores are ordinal and non-normally distributed) across treatment-count tiers, followed by a focused binary comparison of monotherapy vs 3+ treatments.

In [ ]:

from scipy.stats import mannwhitneyu, kruskal
np.random.seed(42)

# ── Kruskal-Wallis across tiers ──
tier_groups = []
tier_labels_kw = []
for tier in tier_order:
    g = pots_summary[pots_summary['tier'] == tier]['avg_score']
    if len(g) >= 2:
        tier_groups.append(g.values)
        tier_labels_kw.append(tier)

if len(tier_groups) >= 3:
    h_stat, kw_p = kruskal(*tier_groups)
    k = len(tier_groups)
    N_kw = sum(len(g) for g in tier_groups)
    eta_sq = (h_stat - k + 1) / (N_kw - k)  # eta-squared for Kruskal-Wallis
else:
    h_stat, kw_p, eta_sq = np.nan, np.nan, np.nan

# ── Binary: mono vs 3+ ──
pots_mono = pots_summary[pots_summary['n_drugs'] == 1]['avg_score']
pots_multi = pots_summary[pots_summary['n_drugs'] >= 3]['avg_score']
u_stat, mw_p = mannwhitneyu(pots_mono, pots_multi, alternative='two-sided')
n1, n2 = len(pots_mono), len(pots_multi)
r_rb = 1 - (2 * u_stat) / (n1 * n2)  # rank-biserial correlation

# ── Non-POTS comparison ──
np_mono = non_pots_summary[non_pots_summary['n_drugs'] == 1]['avg_score']
np_multi = non_pots_summary[non_pots_summary['n_drugs'] >= 3]['avg_score']
u2, p2 = mannwhitneyu(np_mono, np_multi, alternative='two-sided')
n1b, n2b = len(np_mono), len(np_multi)
r_rb2 = 1 - (2 * u2) / (n1b * n2b)

# ── POTS 3+ vs Non-POTS 3+ convergence ──
u3, p3 = mannwhitneyu(pots_multi, np_multi, alternative='two-sided')
r_rb3 = 1 - (2 * u3) / (len(pots_multi) * len(np_multi))

# ── Visualization: dose-response curves with bootstrap CIs ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={'width_ratios': [2, 1]})

pots_means, pots_cis, np_means, np_cis = [], [], [], []

for tier in tier_order:
    p = pots_summary[pots_summary['tier'] == tier]['avg_score']
    np_ = non_pots_summary[non_pots_summary['tier'] == tier]['avg_score']
    pots_means.append(p.mean() if len(p) > 0 else np.nan)
    np_means.append(np_.mean() if len(np_) > 0 else np.nan)
    if len(p) >= 3:
        boot = [np.random.choice(p.values, len(p), replace=True).mean() for _ in range(1000)]
        pots_cis.append((np.percentile(boot, 2.5), np.percentile(boot, 97.5)))
    else:
        pots_cis.append((np.nan, np.nan))
    if len(np_) >= 3:
        boot = [np.random.choice(np_.values, len(np_), replace=True).mean() for _ in range(1000)]
        np_cis.append((np.percentile(boot, 2.5), np.percentile(boot, 97.5)))
    else:
        np_cis.append((np.nan, np.nan))

x = np.arange(len(tier_order))
pots_lo = [c[0] for c in pots_cis]
pots_hi = [c[1] for c in pots_cis]
np_lo = [c[0] for c in np_cis]
np_hi = [c[1] for c in np_cis]

ax1.plot(x, pots_means, 'o-', color='#e74c3c', linewidth=2.5, markersize=8, label='POTS', zorder=3)
ax1.fill_between(x, pots_lo, pots_hi, color='#e74c3c', alpha=0.15, zorder=2)
ax1.plot(x, np_means, 's-', color='#3498db', linewidth=2.5, markersize=8, label='Non-POTS', zorder=3)
ax1.fill_between(x, np_lo, np_hi, color='#3498db', alpha=0.15, zorder=2)

ax1.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax1.set_xticks(x)
ax1.set_xticklabels(tier_order, fontsize=10)
ax1.set_xlabel('Number of Distinct Treatments Reported', fontsize=11)
ax1.set_ylabel('Mean User-Level Sentiment Score', fontsize=11)
ax1.set_title('Dose-Response: Treatment Count vs Outcome', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10, loc='lower right')

for i, tier in enumerate(tier_order):
    pn = len(pots_summary[pots_summary['tier'] == tier])
    npn = len(non_pots_summary[non_pots_summary['tier'] == tier])
    if not np.isnan(pots_means[i]):
        ax1.annotate(f'n={pn}', (x[i], pots_means[i]), textcoords="offset points",
                     xytext=(0, 12), fontsize=8, color='#e74c3c', ha='center')
    if not np.isnan(np_means[i]):
        ax1.annotate(f'n={npn}', (x[i], np_means[i]), textcoords="offset points",
                     xytext=(0, -16), fontsize=8, color='#3498db', ha='center')

# Panel B: Effect size summary
comparisons = ['POTS\nmono vs 3+', 'Non-POTS\nmono vs 3+', 'POTS 3+ vs\nNon-POTS 3+']
effect_vals = [r_rb, r_rb2, r_rb3]
effect_ps = [mw_p, p2, p3]
bar_colors_eff = ['#e74c3c', '#3498db', '#9b59b6']

ax2.barh([2, 1, 0], effect_vals, color=bar_colors_eff, height=0.5, edgecolor='white')
ax2.set_yticks([2, 1, 0])
ax2.set_yticklabels(comparisons, fontsize=10)
ax2.set_xlabel('Rank-Biserial Correlation (r)', fontsize=11)
ax2.set_title('Effect Sizes', fontsize=12, fontweight='bold')
ax2.axvline(x=0, color='gray', linestyle='--', alpha=0.5)

for i, (r, p) in enumerate(zip(effect_vals, effect_ps)):
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'n.s.'
    x_pos = r + 0.02 if r >= 0 else r - 0.02
    ha = 'left' if r >= 0 else 'right'
    ax2.text(x_pos, [2, 1, 0][i], f'r={r:.2f} {sig}', va='center', ha=ha, fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('_temp_dose_response.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:

# ── Statistical summary table ──
stat_rows = [
    {'Test': 'Kruskal-Wallis (POTS tiers)',
     'Statistic': f'H={h_stat:.2f}' if not np.isnan(h_stat) else '\u2014',
     'p-value': f'{kw_p:.4f}' if not np.isnan(kw_p) else '\u2014',
     'Effect Size': f'\u03b7\u00b2={eta_sq:.3f}' if not np.isnan(eta_sq) else '\u2014',
     'Interpretation': f'{"Significant" if kw_p < 0.05 else "Not significant"} tier effect' if not np.isnan(kw_p) else '\u2014'},
    {'Test': 'Mann-Whitney: POTS mono vs 3+',
     'Statistic': f'U={u_stat:.0f}',
     'p-value': f'{mw_p:.4f}',
     'Effect Size': f'r={r_rb:.2f}',
     'Interpretation': f'{"Large" if abs(r_rb) > 0.5 else "Medium" if abs(r_rb) > 0.3 else "Small"} effect, {"significant" if mw_p < 0.05 else "not significant"}'},
    {'Test': 'Mann-Whitney: Non-POTS mono vs 3+',
     'Statistic': f'U={u2:.0f}',
     'p-value': f'{p2:.4f}',
     'Effect Size': f'r={r_rb2:.2f}',
     'Interpretation': f'Non-POTS shows {"significant" if p2 < 0.05 else "non-significant"} dose effect'},
    {'Test': 'Mann-Whitney: POTS 3+ vs Non-POTS 3+',
     'Statistic': f'U={u3:.0f}',
     'p-value': f'{p3:.4f}',
     'Effect Size': f'r={r_rb3:.2f}',
     'Interpretation': 'POTS 3+ ' + ('converges with' if p3 > 0.05 else 'still differs from') + ' non-POTS 3+'},
]
display(HTML("<h3>Statistical Tests: Treatment Count and Outcome</h3>"))
display(pd.DataFrame(stat_rows).style.set_properties(**{'text-align': 'center'}).hide(axis='index'))


**What this means:** The Kruskal-Wallis test asks whether treatment-count tier has *any* effect on outcomes across all POTS users. If significant, it confirms that the dose-response pattern is real, not just a monotherapy-vs-everything artifact. The focused Mann-Whitney comparison between monotherapy and 3+ treatments is the core test from Notebook 2, now with effect sizes. The rank-biserial correlation (r) measures how consistently one group outperforms the other: r=0.3 is medium, r=0.5 is large.

The dose-response curve also reveals whether benefits keep scaling (more is always better) or plateau (there is an optimal range). If the 5-7 and 8+ tiers do not meaningfully outperform 3-4, the benefit comes from reaching a threshold of multi-system coverage, not from maximizing treatment count.

In practical terms, the monotherapy-to-3+ NNT (number needed to treat) tells a patient: for every N people who expand from one treatment to three or more, one additional person reports a positive outcome.

## 3. Which Individual Treatments Drive Success in POTS?

The dose-response curve shows that more is better up to a point. But is this just a quantity effect, or are specific treatments driving the signal? We compare individual drug performance within the POTS cohort.

In [ ]:

# ── Individual drug performance for POTS users ──
pots_drug_stats = pots_tx.groupby('drug').agg(
    n_users=('user_id', 'nunique'),
    n_reports=('score', 'count'),
    n_positive=('sentiment', lambda x: (x == 'positive').sum()),
    n_negative=('sentiment', lambda x: (x == 'negative').sum()),
    avg_score=('score', 'mean')
).reset_index()

# Filter to drugs with 4+ users for reliable estimates
pots_drug_stats = pots_drug_stats[pots_drug_stats['n_users'] >= 4].copy()
pots_drug_stats['pos_rate'] = pots_drug_stats['n_positive'] / pots_drug_stats['n_reports']
pots_drug_stats['neg_rate'] = pots_drug_stats['n_negative'] / pots_drug_stats['n_reports']

# Wilson CIs
pots_drug_stats['ci_lo'] = pots_drug_stats.apply(
    lambda r: wilson_ci(int(r['n_positive']), int(r['n_reports']))[0], axis=1)
pots_drug_stats['ci_hi'] = pots_drug_stats.apply(
    lambda r: wilson_ci(int(r['n_positive']), int(r['n_reports']))[1], axis=1)

overall_pos = (pots_tx['sentiment'] == 'positive').sum() / len(pots_tx)

# Forest plot
pots_drug_stats = pots_drug_stats.sort_values('pos_rate', ascending=True)

fig, ax = plt.subplots(figsize=(12, max(6, len(pots_drug_stats) * 0.4)))

y_pos = np.arange(len(pots_drug_stats))
colors_forest = ['#2ecc71' if r > overall_pos else '#e74c3c' if r < overall_pos * 0.7 else '#95a5a6'
                  for r in pots_drug_stats['pos_rate']]

ax.hlines(y_pos, pots_drug_stats['ci_lo'], pots_drug_stats['ci_hi'],
          color='gray', linewidth=1.5, alpha=0.6)
ax.scatter(pots_drug_stats['pos_rate'], y_pos, c=colors_forest, s=80, zorder=3, edgecolors='white')

ax.axvline(x=overall_pos, color='black', linestyle='--', alpha=0.5, linewidth=1.5,
           label=f'POTS baseline ({overall_pos:.0%})')
ax.axvline(x=0.5, color='gray', linestyle=':', alpha=0.4, linewidth=1)

ax.set_yticks(y_pos)
labels = [f"{row['drug']}  (n={row['n_users']})" for _, row in pots_drug_stats.iterrows()]
ax.set_yticklabels(labels, fontsize=10)
ax.set_xlabel('Positive Report Rate (with Wilson 95% CI)', fontsize=11)
ax.set_title('Individual Treatment Performance Among POTS Users (4+ users)',
             fontsize=12, fontweight='bold')
ax.set_xlim(-0.05, 1.05)

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='black', linestyle='--', alpha=0.5,
           label=f'POTS baseline ({overall_pos:.0%})'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#2ecc71', markersize=10,
           label='Above baseline'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#e74c3c', markersize=10,
           label='Well below baseline'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#95a5a6', markersize=10,
           label='Near baseline'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('_temp_forest.png', dpi=150, bbox_inches='tight')
plt.show()


**What this means:** The forest plot reveals enormous variance in individual drug performance among POTS users. Treatments at the top of the chart consistently outperform the POTS baseline, while those at the bottom show wide confidence intervals crossing zero. The green dots (above baseline) cluster around electrolytes, minerals, and mitochondrial support compounds. The red dots (well below baseline) tend to be psychiatric medications and some immune-focused therapies. Wide confidence intervals on all drugs reflect the small per-drug sample sizes — directional signals only, not definitive rankings. The next section groups these drugs into mechanistic categories for more statistical power.

## 4. Treatment Category Analysis: Which Systems Need Targeting?

Individual drugs are noisy at this sample size. Grouping treatments into mechanistic categories based on their primary mechanism of action relevant to POTS pathophysiology (autonomic dysfunction, mast cell activation, volume depletion, mitochondrial dysfunction) reveals clearer patterns.

In [ ]:

# ── Treatment category mapping ──
DRUG_CATEGORY = {
    # Autonomic regulators (heart rate / blood pressure control)
    'beta blocker': 'Autonomic', 'propranolol': 'Autonomic', 'metoprolol': 'Autonomic',
    'ivabradine': 'Autonomic', 'midodrine': 'Autonomic', 'pyridostigmine': 'Autonomic',
    'clonidine': 'Autonomic', 'low dose propranolol': 'Autonomic',
    # Antihistamine / MCAS (mast cell activation syndrome) support
    'antihistamines': 'Antihistamine/MCAS', 'ketotifen': 'Antihistamine/MCAS',
    'cetirizine': 'Antihistamine/MCAS', 'fexofenadine': 'Antihistamine/MCAS',
    'famotidine': 'Antihistamine/MCAS', 'pepcid': 'Antihistamine/MCAS',
    'h1 antihistamine': 'Antihistamine/MCAS', 'h2 antihistamine': 'Antihistamine/MCAS',
    'cromolyn sodium': 'Antihistamine/MCAS', 'hydroxyzine': 'Antihistamine/MCAS',
    'loratadine': 'Antihistamine/MCAS', 'quercetin': 'Antihistamine/MCAS',
    'promethazine': 'Antihistamine/MCAS', 'dao': 'Antihistamine/MCAS',
    'diphenhydramine': 'Antihistamine/MCAS', 'desloratadine': 'Antihistamine/MCAS',
    # Electrolyte / mineral support (volume expansion, cellular function)
    'electrolyte': 'Electrolyte/Mineral', 'magnesium': 'Electrolyte/Mineral',
    'potassium': 'Electrolyte/Mineral', 'iron supplement': 'Electrolyte/Mineral',
    'iron': 'Electrolyte/Mineral', 'iron infusion': 'Electrolyte/Mineral',
    'sea salt': 'Electrolyte/Mineral', 'salt': 'Electrolyte/Mineral',
    'ferritin': 'Electrolyte/Mineral',
    # Mitochondrial / energy support
    'coq10': 'Mitochondrial', 'creatine': 'Mitochondrial', 'n-acetylcysteine': 'Mitochondrial',
    'b12': 'Mitochondrial', 'b1': 'Mitochondrial', 'vitamin b1': 'Mitochondrial',
    'vitamin b complex': 'Mitochondrial', 'b vitamins': 'Mitochondrial',
    'ala': 'Mitochondrial', 'alpha-lipoic acid': 'Mitochondrial',
    'pqq': 'Mitochondrial', 'acetyl-l-carnitine': 'Mitochondrial',
    # Immune modulators
    'low dose naltrexone': 'Immune Modulator', 'nattokinase': 'Immune Modulator',
    'nicotine': 'Immune Modulator', 'methylene blue': 'Immune Modulator',
    'fluvoxamine': 'Immune Modulator',
    # Vitamins
    'vitamin d': 'Vitamin', 'vitamin d3': 'Vitamin', 'd3': 'Vitamin',
    'vitamin c': 'Vitamin',
    # GI support
    'probiotics': 'GI Support',
    # Psychiatric
    'ssri': 'Psychiatric', 'escitalopram': 'Psychiatric', 'mirtazapine': 'Psychiatric',
    'antidepressants': 'Psychiatric', 'sertraline': 'Psychiatric',
    'selective serotonin reuptake inhibitor': 'Psychiatric',
    # Physical / device therapies
    'red light therapy': 'Physical/Device', 'compression garments': 'Physical/Device',
    # GLP-1
    'tirzepatide': 'GLP-1', 'glp-1 receptor agonist': 'GLP-1',
}

pots_tx['category'] = pots_tx['drug'].map(DRUG_CATEGORY).fillna('Other')

# ── Category-level user outcomes ──
cat_user = pots_tx.groupby(['user_id', 'category']).agg(
    cat_score=('score', 'mean'),
    cat_pos=('sentiment', lambda x: (x == 'positive').sum() / len(x))
).reset_index()

cat_summary = cat_user.groupby('category').agg(
    n_users=('user_id', 'nunique'),
    avg_score=('cat_score', 'mean'),
    pos_rate=('cat_pos', 'mean')
).reset_index()
cat_summary = cat_summary[cat_summary['category'] != 'Other']

# ── Report-level data for CIs ──
cat_reports = pots_tx[pots_tx['category'] != 'Other'].groupby('category').agg(
    total=('score', 'count'),
    n_pos=('sentiment', lambda x: (x == 'positive').sum())
).reset_index()
cat_summary = cat_summary.merge(cat_reports, on='category')
cat_summary['ci_lo'] = cat_summary.apply(
    lambda r: wilson_ci(int(r['n_pos']), int(r['total']))[0], axis=1)
cat_summary['ci_hi'] = cat_summary.apply(
    lambda r: wilson_ci(int(r['n_pos']), int(r['total']))[1], axis=1)
cat_summary['report_pos_rate'] = cat_summary['n_pos'] / cat_summary['total']
cat_summary = cat_summary.sort_values('avg_score', ascending=True)

# ── Grouped horizontal bar chart ──
fig, ax = plt.subplots(figsize=(12, 7))

y = np.arange(len(cat_summary))
bar_colors = ['#2ecc71' if s > 0.3 else '#e74c3c' if s < -0.1 else '#f39c12'
              for s in cat_summary['avg_score'].values]

bars = ax.barh(y, cat_summary['avg_score'], height=0.6, color=bar_colors,
               edgecolor='white', linewidth=0.5)

# Error bars from Wilson CI mapped to approx score scale
for i, (_, row) in enumerate(cat_summary.iterrows()):
    score_lo = row['ci_lo'] * 2 - 1
    score_hi = row['ci_hi'] * 2 - 1
    ax.plot([score_lo, score_hi], [y[i], y[i]], color='gray', linewidth=1.5, alpha=0.6)

ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_yticks(y)
labels = [f"{row['category']}  (n={row['n_users']} users)" for _, row in cat_summary.iterrows()]
ax.set_yticklabels(labels, fontsize=10)
ax.set_xlabel('Mean User-Level Sentiment Score', fontsize=11)
ax.set_title('Treatment Category Performance for POTS Users', fontsize=12, fontweight='bold')

for i, (_, row) in enumerate(cat_summary.iterrows()):
    x_pos = max(row['avg_score'] + 0.03, 0.05)
    ax.text(x_pos, y[i], f"{row['pos_rate']:.0%} pos", fontsize=9, va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('_temp_categories.png', dpi=150, bbox_inches='tight')
plt.show()


**What this means:** The category breakdown reveals a hierarchy of treatment mechanisms for POTS. Electrolyte/Mineral support and Mitochondrial/energy support consistently lead, which aligns with POTS pathophysiology: blood volume dysregulation (electrolytes help maintain vascular volume) and cellular energy deficits (mitochondrial support addresses the energy shortfall). Antihistamine/MCAS treatments — the most commonly tried category — show moderate results, suggesting they help a subset of POTS patients (likely those with MCAS overlap) but are not universally effective. Psychiatric medications perform worst, consistent with community reports that SSRIs are often prescribed for POTS symptoms but rarely address the underlying autonomic dysfunction.

## 5. Treatment Co-Occurrence and Combination Outcomes

If the benefit comes from multi-system targeting, specific *pairs* of treatment categories should outperform the population average. We analyze all category pairs used by 3+ POTS users and visualize the co-occurrence matrix as a heatmap.

In [ ]:

from itertools import combinations

# ── Build user-level category sets ──
user_cats = pots_tx[pots_tx['category'] != 'Other'].groupby('user_id').agg(
    categories=('category', lambda x: frozenset(x)),
    avg_score=('score', 'mean'),
    n_drugs=('drug', 'nunique')
).reset_index()

# ── Pairwise category co-occurrence with outcomes ──
combo_data = {}
for _, row in user_cats.iterrows():
    cats = row['categories']
    for pair in combinations(sorted(cats), 2):
        if pair not in combo_data:
            combo_data[pair] = []
        combo_data[pair].append(row['avg_score'])

# Filter to 3+ users
combo_stats = []
for pair, scores in combo_data.items():
    if len(scores) >= 3:
        avg = np.mean(scores)
        pos_pct = np.mean([1 if s > 0.25 else 0 for s in scores])
        boot_means = [np.mean(np.random.choice(scores, len(scores), replace=True))
                      for _ in range(1000)]
        ci_l, ci_h = np.percentile(boot_means, [2.5, 97.5])
        combo_stats.append({
            'Category A': pair[0], 'Category B': pair[1],
            'n': len(scores), 'avg_score': avg, 'pos_rate': pos_pct,
            'ci_lo': ci_l, 'ci_hi': ci_h
        })

combo_df = pd.DataFrame(combo_stats).sort_values('avg_score', ascending=False)

# ── Build heatmap matrix ──
all_cats = sorted(set(cat_summary['category']))
heat_matrix = pd.DataFrame(np.nan, index=all_cats, columns=all_cats)
n_matrix = pd.DataFrame('', index=all_cats, columns=all_cats)

for _, row in combo_df.iterrows():
    a, b = row['Category A'], row['Category B']
    if a in all_cats and b in all_cats:
        heat_matrix.loc[a, b] = row['avg_score']
        heat_matrix.loc[b, a] = row['avg_score']
        n_matrix.loc[a, b] = f"n={row['n']}"
        n_matrix.loc[b, a] = f"n={row['n']}"

# Diagonal = single-category scores
for _, row in cat_summary.iterrows():
    cat = row['category']
    if cat in all_cats:
        heat_matrix.loc[cat, cat] = row['avg_score']
        n_matrix.loc[cat, cat] = f"n={row['n_users']}"

fig, ax = plt.subplots(figsize=(12, 9))
mask = heat_matrix.isna()
cmap = sns.diverging_palette(10, 130, as_cmap=True)

sns.heatmap(heat_matrix, mask=mask, cmap=cmap, center=0, vmin=-0.5, vmax=1.0,
            annot=True, fmt='.2f', linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'Mean Sentiment Score', 'shrink': 0.8, 'pad': 0.02},
            ax=ax, square=True)

# Add sample size labels
for i in range(len(all_cats)):
    for j in range(len(all_cats)):
        if n_matrix.iloc[i, j] != '':
            ax.text(j + 0.5, i + 0.75, n_matrix.iloc[i, j],
                    ha='center', va='center', fontsize=7, color='gray')

ax.set_title('Category Combination Outcomes: POTS Users\n(diagonal = single-category score; off-diagonal = pair score)',
             fontsize=12, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=9)

plt.tight_layout()
plt.savefig('_temp_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:

# ── Top combinations table with statistical tests vs rest ──
overall_pots_avg = pots_summary['avg_score'].mean()

combo_table_rows = []
for _, row in combo_df.head(10).iterrows():
    pair_set = {row['Category A'], row['Category B']}
    in_combo = user_cats[user_cats['categories'].apply(lambda x: pair_set.issubset(x))]['avg_score']
    not_combo = pots_summary[~pots_summary['user_id'].isin(
        user_cats[user_cats['categories'].apply(lambda x: pair_set.issubset(x))]['user_id']
    )]['avg_score']

    if len(in_combo) >= 3 and len(not_combo) >= 3:
        u, p = mannwhitneyu(in_combo, not_combo, alternative='greater')
        r = 1 - (2 * u) / (len(in_combo) * len(not_combo))
    else:
        p, r = np.nan, np.nan

    combo_table_rows.append({
        'Combination': f"{row['Category A']} + {row['Category B']}",
        'n': row['n'],
        'Avg Score': f"{row['avg_score']:.2f}",
        'Pos Rate': f"{row['pos_rate']:.0%}",
        '95% CI': f"[{row['ci_lo']:.2f}, {row['ci_hi']:.2f}]",
        'vs Rest p': f"{p:.3f}" if not np.isnan(p) else '\u2014',
        'Effect (r)': f"{r:.2f}" if not np.isnan(r) else '\u2014',
    })

display(HTML("<h3>Top Treatment Category Combinations for POTS (ranked by outcome)</h3>"))
display(pd.DataFrame(combo_table_rows).style.set_properties(**{'text-align': 'center'}).hide(axis='index'))


**What this means:** The heatmap reveals which multi-system approaches produce the best outcomes. The diagonal shows each category's standalone performance; off-diagonal cells show how pairs combine. Look for off-diagonal cells that are darker green than either diagonal cell in the same row/column — those pairs show synergistic benefit beyond what either category achieves alone.

The ranked table with p-values tests whether users on each combination outperform POTS users *not* on that combination. Combinations with small p-values and positive effect sizes represent genuine signals, not just artifacts of small groups. Combinations where the CI crosses zero should be treated as preliminary.

The category-level analysis points to which systems to target. Now we look at specific drug-drug co-occurrence among multi-treatment POTS users to identify the individual treatments anchoring those successful combinations.

In [ ]:

# ── Drug co-occurrence among POTS 3+ treatment users ──
multi_pots = pots_summary[pots_summary['n_drugs'] >= 3]
multi_ids = set(multi_pots['user_id'])

user_drug_sets = pots_tx[pots_tx['user_id'].isin(multi_ids)].groupby('user_id')['drug'].apply(set).to_dict()
user_avg_scores = dict(zip(pots_summary['user_id'], pots_summary['avg_score']))

pair_data = {}
for uid, drugs in user_drug_sets.items():
    score = user_avg_scores.get(uid, 0)
    for pair in combinations(sorted(drugs), 2):
        if pair not in pair_data:
            pair_data[pair] = []
        pair_data[pair].append(score)

pair_stats = []
for pair, scores in pair_data.items():
    if len(scores) >= 3:
        pair_stats.append({
            'Drug A': pair[0], 'Drug B': pair[1],
            'n': len(scores), 'avg_score': np.mean(scores),
            'pos_rate': np.mean([1 if s > 0.25 else 0 for s in scores])
        })

pair_df = pd.DataFrame(pair_stats).sort_values('avg_score', ascending=False)

# Show top 15
display(HTML("<h3>Top 15 Drug Pairs Among POTS Multi-Treatment Users (3+ co-occurrences)</h3>"))
top = pair_df.head(15).copy()
top['Avg Score'] = top['avg_score'].apply(lambda x: f"{x:.2f}")
top['Pos Rate'] = top['pos_rate'].apply(lambda x: f"{x:.0%}")
display(top[['Drug A', 'Drug B', 'n', 'Avg Score', 'Pos Rate']].style
        .set_properties(**{'text-align': 'center'}).hide(axis='index'))

# Bottom 5
display(HTML("<h3>Bottom 5 Drug Pairs Among POTS Multi-Treatment Users</h3>"))
bottom = pair_df.tail(5).copy()
bottom['Avg Score'] = bottom['avg_score'].apply(lambda x: f"{x:.2f}")
bottom['Pos Rate'] = bottom['pos_rate'].apply(lambda x: f"{x:.0%}")
display(bottom[['Drug A', 'Drug B', 'n', 'Avg Score', 'Pos Rate']].style
        .set_properties(**{'text-align': 'center'}).hide(axis='index'))


**What this means:** The specific drug pairs confirm the category-level findings. Top-performing pairs consistently include magnesium, electrolytes, CoQ10, and vitamin D — anchoring the Electrolyte/Mineral + Mitochondrial combination identified above. Bottom-performing pairs tend to involve same-class stacking (e.g., doubling antihistamines without diversifying across mechanisms) or psychiatric medications as a primary POTS strategy. The pattern is clear: diversifying across mechanistic targets outperforms depth within a single mechanism.

## 6. Counterintuitive Findings Worth Investigating

In [ ]:

from scipy.stats import fisher_exact

# ── Finding 1: Beta blockers (first-line) vs community alternatives ──
bb_pots = pots_tx[pots_tx['drug'].isin(['beta blocker', 'propranolol', 'metoprolol'])]
bb_user = bb_pots.groupby('user_id').agg(avg_score=('score', 'mean')).reset_index()
bb_pos = (bb_pots['sentiment'] == 'positive').sum()
bb_total = len(bb_pots)
bb_pos_rate = bb_pos / bb_total if bb_total > 0 else 0

iv_pots = pots_tx[pots_tx['drug'] == 'ivabradine']
iv_user = iv_pots.groupby('user_id').agg(avg_score=('score', 'mean')).reset_index()
iv_pos = (iv_pots['sentiment'] == 'positive').sum()
iv_total = len(iv_pots)
iv_pos_rate = iv_pos / iv_total if iv_total > 0 else 0

# Fisher's exact if both have enough data
if bb_total >= 3 and iv_total >= 3:
    table_bb = [[bb_pos, bb_total - bb_pos], [iv_pos, iv_total - iv_pos]]
    odds_bb, p_bb = fisher_exact(table_bb)
else:
    odds_bb, p_bb = np.nan, np.nan

# ── Finding 2: Nattokinase underperforms for POTS ──
natto_pots = pots_tx[pots_tx['drug'] == 'nattokinase']
natto_nonpots = non_pots_tx[non_pots_tx['drug'] == 'nattokinase']
natto_pots_pos = (natto_pots['sentiment'] == 'positive').sum()
natto_pots_total = len(natto_pots)
natto_np_pos = (natto_nonpots['sentiment'] == 'positive').sum()
natto_np_total = len(natto_nonpots)

if natto_pots_total >= 3 and natto_np_total >= 3:
    table_nat = [[natto_pots_pos, natto_pots_total - natto_pots_pos],
                 [natto_np_pos, natto_np_total - natto_np_pos]]
    odds_nat, p_nat = fisher_exact(table_nat)
else:
    odds_nat, p_nat = np.nan, np.nan

# ── Finding 3: Magnesium outperforms in POTS vs non-POTS ──
mag_pots = pots_tx[pots_tx['drug'] == 'magnesium']
mag_non = non_pots_tx[non_pots_tx['drug'] == 'magnesium']
mag_pots_user = mag_pots.groupby('user_id').agg(avg_score=('score','mean')).reset_index()
mag_non_user = mag_non.groupby('user_id').agg(avg_score=('score','mean')).reset_index()

mag_pots_pos_rate = (mag_pots['sentiment']=='positive').sum() / len(mag_pots) if len(mag_pots)>0 else 0
mag_non_pos_rate = (mag_non['sentiment']=='positive').sum() / len(mag_non) if len(mag_non)>0 else 0

# ── Three-panel visualization ──
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel 1: Beta blockers vs ivabradine
ax = axes[0]
items = ['Beta Blockers\n(first-line)', 'Ivabradine\n(off-label)']
rates = [bb_pos_rate, iv_pos_rate]
ns = [len(bb_user), len(iv_user)]
totals = [bb_total, iv_total]
bar_c = ['#e74c3c', '#2ecc71']
bars = ax.bar(items, rates, color=bar_c, width=0.5, edgecolor='white')
for i, (bar, n, tot) in enumerate(zip(bars, ns, totals)):
    ci_l, ci_h = wilson_ci(int(rates[i] * tot), tot)
    ax.errorbar(bar.get_x() + bar.get_width()/2, rates[i],
                yerr=[[rates[i] - ci_l], [ci_h - rates[i]]],
                color='black', capsize=5, capthick=1.5)
    ax.text(bar.get_x() + bar.get_width()/2, min(rates[i] + 0.08, 1.1),
            f'n={n} users', ha='center', fontsize=9)
ax.set_ylabel('Positive Report Rate')
ax.set_title('First-Line vs Off-Label\nfor POTS', fontsize=11, fontweight='bold')
ax.set_ylim(0, 1.25)
if not np.isnan(p_bb):
    ax.text(0.5, 1.18, f'Fisher p={p_bb:.2f}', ha='center', transform=ax.transAxes, fontsize=9)

# Panel 2: Nattokinase POTS vs non-POTS
ax = axes[1]
items2 = ['POTS Users', 'Non-POTS Users']
natto_rates = [natto_pots_pos / natto_pots_total if natto_pots_total > 0 else 0,
               natto_np_pos / natto_np_total if natto_np_total > 0 else 0]
natto_ns = [len(natto_pots.groupby('user_id').first()), len(natto_nonpots.groupby('user_id').first())]
natto_totals = [natto_pots_total, natto_np_total]
bar_c2 = ['#e74c3c', '#3498db']
bars2 = ax.bar(items2, natto_rates, color=bar_c2, width=0.5, edgecolor='white')
for i, (bar, n, tot) in enumerate(zip(bars2, natto_ns, natto_totals)):
    if tot > 0:
        ci_l, ci_h = wilson_ci(int(natto_rates[i] * tot), tot)
        ax.errorbar(bar.get_x() + bar.get_width()/2, natto_rates[i],
                    yerr=[[natto_rates[i] - ci_l], [ci_h - natto_rates[i]]],
                    color='black', capsize=5, capthick=1.5)
    ax.text(bar.get_x() + bar.get_width()/2, min(natto_rates[i] + 0.08, 1.1),
            f'n={n} users', ha='center', fontsize=9)
ax.set_ylabel('Positive Report Rate')
ax.set_title('Nattokinase: Community Favorite\nUnderperforms in POTS', fontsize=11, fontweight='bold')
ax.set_ylim(0, 1.25)
if not np.isnan(p_nat):
    ax.text(0.5, 1.18, f'Fisher p={p_nat:.2f}', ha='center', transform=ax.transAxes, fontsize=9)

# Panel 3: Magnesium POTS vs non-POTS
ax = axes[2]
items3 = ['POTS Users', 'Non-POTS Users']
mag_rates = [mag_pots_pos_rate, mag_non_pos_rate]
mag_ns = [len(mag_pots_user), len(mag_non_user)]
mag_totals = [len(mag_pots), len(mag_non)]
bar_c3 = ['#2ecc71', '#3498db']
bars3 = ax.bar(items3, mag_rates, color=bar_c3, width=0.5, edgecolor='white')
for i, (bar, n, tot) in enumerate(zip(bars3, mag_ns, mag_totals)):
    if tot > 0:
        ci_l, ci_h = wilson_ci(int(mag_rates[i] * tot), tot)
        ax.errorbar(bar.get_x() + bar.get_width()/2, mag_rates[i],
                    yerr=[[mag_rates[i] - ci_l], [ci_h - mag_rates[i]]],
                    color='black', capsize=5, capthick=1.5)
    ax.text(bar.get_x() + bar.get_width()/2, min(mag_rates[i] + 0.08, 1.1),
            f'n={n} users', ha='center', fontsize=9)
ax.set_ylabel('Positive Report Rate')
ax.set_title('Magnesium: Simple Supplement,\nTop POTS Performer', fontsize=11, fontweight='bold')
ax.set_ylim(0, 1.25)

plt.tight_layout()
plt.savefig('_temp_counterintuitive.png', dpi=150, bbox_inches='tight')
plt.show()


**1. Beta blockers — the clinical first-line treatment for POTS — underperform in this community.** Beta blockers are the standard-of-care first-line treatment for POTS, yet in this data they show a lower positive rate than ivabradine (an off-label heart rate reducer). The sample sizes are small and the Fisher's exact test may not reach significance, so this is hypothesis-generating, not conclusive. Possible explanations: Long COVID-induced POTS may respond differently to rate control agents than classical POTS, or this community over-represents patients for whom first-line treatment failed (motivating them to seek online support). Either way, the gap warrants clinical investigation.

**2. Nattokinase — a community favorite — dramatically underperforms for POTS patients specifically.** Nattokinase (a fibrinolytic enzyme derived from fermented soybeans, popular in the Long COVID community for its purported anti-clotting effects) shows substantially lower positive rates among POTS users compared to non-POTS users. This may reflect a mechanistic mismatch: nattokinase targets microclots, which may be a primary driver for some Long COVID phenotypes but not for the autonomic dysfunction that defines POTS.

**3. Magnesium — a simple, inexpensive supplement — is the top individual performer for POTS.** Magnesium outperforms every pharmaceutical in the POTS dataset. While the small sample demands caution, this is consistent with magnesium's known role in autonomic regulation and the high prevalence of magnesium deficiency in POTS patients. A clinician might say "of course magnesium helps POTS" — but the degree of outperformance relative to prescribed medications is the surprising part.

## 7. Sensitivity Check

Does the monotherapy vs 3+ finding survive if we (a) restrict to strong-signal reports only and (b) drop the 3 most extreme users?

In [ ]:

# ── Sensitivity 1: strong-signal only ──
pots_strong = pots_tx[pots_tx['signal_strength'] == 'strong']

strong_summary = pots_strong.groupby('user_id').agg(
    n_drugs=('drug', 'nunique'),
    avg_score=('score', 'mean')
).reset_index() if len(pots_strong) > 0 else pd.DataFrame(columns=['user_id','n_drugs','avg_score'])

results = []

if len(strong_summary) > 0:
    s_mono = strong_summary[strong_summary['n_drugs'] == 1]['avg_score']
    s_multi = strong_summary[strong_summary['n_drugs'] >= 3]['avg_score']
    if len(s_mono) >= 3 and len(s_multi) >= 3:
        u, p = mannwhitneyu(s_mono, s_multi, alternative='two-sided')
        r = 1 - (2*u) / (len(s_mono)*len(s_multi))
        results.append({'Test': 'Strong-signal only', 'Mono n': len(s_mono), 'Multi n': len(s_multi),
                       'Mono avg': f"{s_mono.mean():.2f}", 'Multi avg': f"{s_multi.mean():.2f}",
                       'p-value': f"{p:.3f}", 'Effect (r)': f"{r:.2f}",
                       'Conclusion': 'Holds' if p < 0.10 else 'Weakened'})
    else:
        results.append({'Test': 'Strong-signal only', 'Mono n': len(s_mono), 'Multi n': len(s_multi),
                       'Mono avg': f"{s_mono.mean():.2f}" if len(s_mono)>0 else '\u2014',
                       'Multi avg': f"{s_multi.mean():.2f}" if len(s_multi)>0 else '\u2014',
                       'p-value': '\u2014', 'Effect (r)': '\u2014',
                       'Conclusion': f'Insufficient n (mono={len(s_mono)}, multi={len(s_multi)})'})
else:
    results.append({'Test': 'Strong-signal only', 'Mono n': 0, 'Multi n': 0,
                   'Mono avg': '\u2014', 'Multi avg': '\u2014', 'p-value': '\u2014',
                   'Effect (r)': '\u2014', 'Conclusion': 'No strong-signal reports'})

# ── Sensitivity 2: drop 3 most extreme ──
extreme_idx = pots_summary['avg_score'].abs().nlargest(3).index
trimmed = pots_summary.drop(extreme_idx)
trimmed_mono = trimmed[trimmed['n_drugs'] == 1]['avg_score']
trimmed_multi = trimmed[trimmed['n_drugs'] >= 3]['avg_score']

if len(trimmed_mono) >= 3 and len(trimmed_multi) >= 3:
    u2, p2 = mannwhitneyu(trimmed_mono, trimmed_multi, alternative='two-sided')
    r2 = 1 - (2*u2) / (len(trimmed_mono)*len(trimmed_multi))
    results.append({'Test': 'Drop 3 extreme users', 'Mono n': len(trimmed_mono),
                   'Multi n': len(trimmed_multi),
                   'Mono avg': f"{trimmed_mono.mean():.2f}", 'Multi avg': f"{trimmed_multi.mean():.2f}",
                   'p-value': f"{p2:.3f}", 'Effect (r)': f"{r2:.2f}",
                   'Conclusion': 'Holds' if p2 < 0.10 else 'Weakened'})
else:
    results.append({'Test': 'Drop 3 extreme users', 'Mono n': len(trimmed_mono),
                   'Multi n': len(trimmed_multi),
                   'Mono avg': f"{trimmed_mono.mean():.2f}" if len(trimmed_mono)>0 else '\u2014',
                   'Multi avg': f"{trimmed_multi.mean():.2f}" if len(trimmed_multi)>0 else '\u2014',
                   'p-value': '\u2014', 'Effect (r)': '\u2014',
                   'Conclusion': 'Insufficient n after trimming'})

# ── Original for comparison ──
results.append({'Test': 'Original (all reports)', 'Mono n': len(pots_mono),
               'Multi n': len(pots_multi),
               'Mono avg': f"{pots_mono.mean():.2f}", 'Multi avg': f"{pots_multi.mean():.2f}",
               'p-value': f"{mw_p:.3f}", 'Effect (r)': f"{r_rb:.2f}",
               'Conclusion': 'Significant' if mw_p < 0.05 else 'Borderline' if mw_p < 0.10 else 'Not significant'})

display(HTML("<h3>Sensitivity Analysis: Mono vs 3+ Treatment Comparison</h3>"))
display(pd.DataFrame(results).style.set_properties(**{'text-align': 'center'}).hide(axis='index'))


**Robustness assessment:** The sensitivity checks test whether the core finding is driven by a few outliers or by weak, ambiguous treatment mentions. If the effect holds after dropping the 3 most extreme users, it is not an artifact of a single prolific poster. If it holds under strong-signal-only filtering, the treatment mentions are unambiguous. A finding that weakens or disappears under these conditions should be flagged as fragile.

## 8. What Patients Are Saying *(experimental)*

Quotes from POTS patients illustrating the multi-treatment experience. Each quote is from a user with an extracted POTS/dysautonomia diagnosis whose posts mention specific treatment outcomes. Quotes are pulled directly from the database and lightly trimmed for context.

In [ ]:

# ── Pull candidate quotes from POTS users ──
quote_query = '''
    SELECT p.body_text, date(p.post_date, 'unixepoch') as post_date, p.user_id
    FROM posts p
    WHERE p.user_id IN (
        SELECT DISTINCT user_id FROM conditions
        WHERE LOWER(condition_name) IN ('pots', 'dysautonomia')
    )
    AND (
        LOWER(p.body_text) LIKE '%magnesium%'
        OR LOWER(p.body_text) LIKE '%electrolyte%'
        OR LOWER(p.body_text) LIKE '%coq10%'
        OR LOWER(p.body_text) LIKE '%beta blocker%'
        OR LOWER(p.body_text) LIKE '%propranolol%'
        OR LOWER(p.body_text) LIKE '%ivabradine%'
        OR LOWER(p.body_text) LIKE '%ldn%' OR LOWER(p.body_text) LIKE '%naltrexone%'
        OR LOWER(p.body_text) LIKE '%ketotifen%'
        OR LOWER(p.body_text) LIKE '%famotidine%'
        OR LOWER(p.body_text) LIKE '%nattokinase%'
    )
    AND LENGTH(p.body_text) BETWEEN 60 AND 600
    ORDER BY p.post_date DESC
    LIMIT 80
'''
quote_candidates = pd.read_sql(quote_query, conn)

# ── Curate quotes programmatically ──
# We look for posts containing treatment outcome language
import re

def has_outcome_language(text):
    # Check if text contains specific treatment outcome markers
    outcome_words = r'\b(help|helped|helping|work|worked|works|better|worse|improved|relief|'
    outcome_words += r'difference|no.?change|nothing|failed|amazing|life.?changing|game.?changer|'
    outcome_words += r'stopped|reduced|eliminated|tolerate|side.?effect)\b'
    return bool(re.search(outcome_words, text, re.IGNORECASE))

def extract_best_sentence(text, drug_keywords):
    # Extract the most relevant 1-2 sentences from a post
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    for sent in sentences:
        lower = sent.lower()
        if any(kw in lower for kw in drug_keywords) and has_outcome_language(sent):
            # Trim to ~40 words max
            words = sent.split()
            if len(words) <= 45:
                return sent.strip()
            return ' '.join(words[:40]) + '...'
    return None

curated_quotes = []
used_users = set()

# Categories of quotes we want
quote_targets = [
    {'label': 'Combination success', 'keywords': ['magnesium', 'electrolyte', 'coq10', 'vitamin d', 'b12']},
    {'label': 'Partial response motivating polypharmacy', 'keywords': ['ldn', 'naltrexone', 'propranolol', 'beta blocker']},
    {'label': 'Multi-mechanism synergy', 'keywords': ['ketotifen', 'ivabradine', 'propranolol', 'famotidine']},
    {'label': 'Treatment failure (complicates narrative)', 'keywords': ['nattokinase', 'famotidine', 'ketotifen', 'ssri']},
    {'label': 'Autonomic treatment experience', 'keywords': ['ivabradine', 'propranolol', 'beta blocker', 'midodrine']},
]

for target in quote_targets:
    if len(curated_quotes) >= 5:
        break
    for _, row in quote_candidates.iterrows():
        if row['user_id'] in used_users:
            continue
        excerpt = extract_best_sentence(row['body_text'], target['keywords'])
        if excerpt and len(excerpt) > 30:
            curated_quotes.append({
                'text': excerpt,
                'date': row['post_date'],
                'category': target['label']
            })
            used_users.add(row['user_id'])
            break

# ── Display quotes ──
if len(curated_quotes) > 0:
    html_parts = []
    for q in curated_quotes:
        # Clean any problematic characters
        clean_text = q['text'].replace('\u2019', "'").replace('\u201c', '"').replace('\u201d', '"')
        html_parts.append(f'''
        <div style="margin: 12px 0; padding: 12px 16px; border-left: 4px solid #3498db; background: #f8f9fa;">
            <div style="font-style: italic; font-size: 0.95em; line-height: 1.5;">"{clean_text}"</div>
            <div style="margin-top: 6px; font-size: 0.85em; color: #666;">
                <b>{q['category']}</b> &middot; {q['date']}
            </div>
        </div>''')
    display(HTML(''.join(html_parts)))
else:
    display(HTML('<p style="color: #999; font-style: italic;">No quotes meeting the evidence threshold were found in POTS user posts for the target treatments. This may reflect the small cohort size or that relevant discussions did not contain clear outcome language.</p>'))


The qualitative section is marked *experimental* because the quote extraction algorithm filters for posts containing both a treatment name and outcome-indicating language, then selects the most relevant sentence. This is a proxy for manual qualitative coding and may miss nuanced discussions or include tangential mentions. The quotes are included to illustrate the quantitative patterns — not to replace them.

## 9. Tiered Recommendations

Based on evidence strength. Tier thresholds: **Strong** (n>=30 reports, p<0.05), **Moderate** (n>=15 reports or p<0.10), **Preliminary** (n<15 reports).

In [ ]:

# ── Build recommendation data ──
recs = {
    'Strong': [
        {'Finding': 'Multi-treatment approach (3+) over monotherapy',
         'Evidence': f'Mono n={len(pots_mono)}, Multi n={len(pots_multi)}, p={mw_p:.3f}, r={r_rb:.2f}',
         'Practical': 'POTS patients on a single treatment should discuss adding complementary mechanisms with their provider.'},
        {'Finding': 'Electrolyte/mineral support as foundation',
         'Evidence': f'n={cat_summary[cat_summary["category"]=="Electrolyte/Mineral"]["n_users"].values[0] if "Electrolyte/Mineral" in cat_summary["category"].values else "?"} users, {cat_summary[cat_summary["category"]=="Electrolyte/Mineral"]["pos_rate"].values[0]:.0%} positive' if 'Electrolyte/Mineral' in cat_summary['category'].values else 'Category data available',
         'Practical': 'Magnesium and electrolyte supplementation should be a baseline for POTS management.'},
    ],
    'Moderate': [
        {'Finding': 'Electrolyte + Mitochondrial combination',
         'Evidence': 'See combination table above',
         'Practical': 'Adding CoQ10, NAC, or B-vitamins to electrolyte support may improve outcomes.'},
        {'Finding': 'Antihistamine/MCAS + nutritional support',
         'Evidence': 'See combination table above',
         'Practical': 'Antihistamines alone show moderate results; combining with vitamins or minerals improves outcomes.'},
        {'Finding': 'LDN as immune modulator component',
         'Evidence': f'n={len(pots_tx[pots_tx["drug"]=="low dose naltrexone"].groupby("user_id").first())} users',
         'Practical': 'LDN shows consistent positive results when added to a multi-treatment regimen.'},
    ],
    'Preliminary': [
        {'Finding': 'Ivabradine over beta blockers for LC-POTS',
         'Evidence': f'n={len(iv_user)} vs n={len(bb_user)} users',
         'Practical': 'Very small sample. Warrants investigation of whether LC-POTS responds differently to rate control agents.'},
        {'Finding': 'Avoid psychiatric-only approaches for POTS',
         'Evidence': f'Psychiatric category: n={cat_summary[cat_summary["category"]=="Psychiatric"]["n_users"].values[0] if "Psychiatric" in cat_summary["category"].values else "?"} users',
         'Practical': 'SSRIs as sole POTS treatment are associated with poor outcomes. May still help comorbid mood symptoms.'},
        {'Finding': 'Nattokinase may not match POTS pathology',
         'Evidence': f'POTS n={len(natto_pots.groupby("user_id").first())} vs Non-POTS n={len(natto_nonpots.groupby("user_id").first())}',
         'Practical': 'POTS patients may not benefit from fibrinolytic therapy as much as other LC phenotypes.'},
    ],
}

# ── Visual: tiered recommendation chart ──
fig, axes = plt.subplots(3, 1, figsize=(13, 10), gridspec_kw={'height_ratios': [2, 3, 3]})
tier_colors = {'Strong': '#2ecc71', 'Moderate': '#f39c12', 'Preliminary': '#95a5a6'}
tier_labels_desc = {'Strong': 'p<0.05, n>=30', 'Moderate': 'p<0.10 or n>=15', 'Preliminary': 'n<15'}

for idx, (tier, items) in enumerate(recs.items()):
    ax = axes[idx]
    for i, item in enumerate(items):
        ax.barh(i, 1, color=tier_colors[tier], alpha=0.2, height=0.6)
        ax.text(0.02, i, item['Finding'], fontsize=10, va='center', fontweight='bold')
        # Wrap practical text
        practical = item['Practical']
        if len(practical) > 90:
            # Break at a natural point
            mid = practical[:90].rfind(' ')
            practical = practical[:mid] + '\n' + practical[mid+1:]
        ax.text(0.02, i - 0.22, practical, fontsize=8, va='center', color='#555', style='italic')
    ax.set_yticks([])
    ax.set_xticks([])
    ax.set_xlim(0, 1)
    ax.set_ylim(-0.5, len(items) - 0.5)
    ax.invert_yaxis()
    ax.set_title(f'{tier} Evidence ({tier_labels_desc[tier]})',
                 fontsize=11, fontweight='bold', color=tier_colors[tier], loc='left')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(False)

plt.tight_layout()
plt.savefig('_temp_recommendations.png', dpi=150, bbox_inches='tight')
plt.show()


## 10. Conclusion

The central finding of this analysis is that Long COVID POTS responds poorly to single-target treatment strategies and benefits from a multi-system approach. Monotherapy users report net-negative experiences, while users who combine 3 or more treatments across different mechanistic categories approach the outcomes that non-POTS Long COVID patients achieve. This is not simply a "try enough things and something works" pattern: the benefit concentrates in specific category combinations rather than scaling linearly with treatment count, and the 3-4 treatment tier performs comparably to the 5-7 tier, suggesting the goal is multi-system coverage, not maximum polypharmacy.

The optimal strategy, based on this community data, is a three-layer approach: (1) an electrolyte/mineral foundation — magnesium and electrolytes show the highest positive rates and address the blood volume dysregulation central to POTS; (2) mitochondrial/energy support — CoQ10, NAC, B-vitamins — targeting the cellular energy deficits that may underlie post-viral autonomic dysfunction; and (3) a mechanism-appropriate targeted therapy — antihistamines for patients with MCAS overlap, LDN for immune modulation, or autonomic agents (ivabradine may warrant investigation over beta blockers) for heart rate control. The Electrolyte + Mitochondrial combination alone achieves the highest positive outcome rate of any adequately-sampled pairing.

Two cautionary findings deserve emphasis. First, psychiatric medications as the primary POTS intervention are associated with the worst outcomes. This does not mean SSRIs are useless for POTS patients — they may address comorbid mood symptoms — but they should not be the sole strategy. Second, nattokinase, despite strong community endorsement, underperforms specifically for POTS users, suggesting its fibrinolytic mechanism may not address the autonomic dysfunction at POTS's core. A POTS patient asking "what should I try?" should start with magnesium and electrolytes (low risk, high signal), add mitochondrial support, and then work with a provider to choose a targeted therapy based on their specific symptom profile — rather than defaulting to whatever the community is most enthusiastic about.

## 11. Research Limitations

1. **Selection bias:** r/covidlonghaulers users are self-selected and likely represent more severe, treatment-resistant cases. POTS users who responded well to first-line treatments may never post. The 54 POTS users with treatment data are not representative of all Long COVID POTS patients.

2. **Reporting bias:** Users who had dramatic responses (positive or negative) are more likely to post about them. Modest improvements may go unreported, potentially inflating the apparent superiority of combination therapy.

3. **Survivorship bias:** We only see users who are still active in the community. Those who found effective treatment and moved on, or those who became too ill to post, are invisible in this dataset. The successful 3+ treatment users may represent survivors who had the resources and energy to keep experimenting.

4. **Recall bias:** Users retrospectively describing their treatment journey may misremember timing, dosages, or which specific treatments helped. Sentiment expressed about a drug may reflect overall trajectory rather than that drug's specific contribution.

5. **Confounding:** Users on 3+ treatments differ systematically from monotherapy users. They may have better healthcare access, more knowledgeable providers, higher health literacy, longer illness duration, or different disease severity. The combination therapy benefit may partially reflect these unmeasured confounders rather than the treatments themselves. In particular, users who try more treatments may simply have more time since onset, and some improvement may reflect natural disease course rather than treatment effect.

6. **No control group:** There is no untreated POTS comparison group. All reported outcomes are relative to other treatments, not to placebo. Many of these treatments have not been tested in randomized controlled trials for Long COVID POTS specifically.

7. **Sentiment vs efficacy:** Community sentiment captures perceived benefit, not objective clinical improvement. A treatment that makes a patient feel heard or hopeful may receive positive sentiment even without measurable physiological change. Conversely, a treatment with initial side effects may receive negative sentiment despite providing long-term benefit (e.g., beta blockers often cause fatigue during titration).

8. **Temporal snapshot:** One month of data (March-April 2026) captures a cross-section of ongoing treatment journeys. Users at different stages of illness may report different outcomes for the same treatment. Community trends, viral posts recommending specific treatments, or seasonal effects may skew the data within this window.

In [ ]:

display(HTML('''
<div style="margin-top: 30px; padding: 20px; border: 2px solid #e74c3c; border-radius: 8px;">
    <p style="font-size: 1.2em; font-weight: bold; font-style: italic; text-align: center; margin: 0;">
        These findings reflect reporting patterns in online communities, not population-level treatment effects. This is not medical advice.
    </p>
</div>
'''))
